In [1]:
ls

README.md   cpp_wrappers/  images/   mapping/   requirements.txt   trainer/
Test.ipynb  datasets/      infer.py  models/    semantickitti.zip  utils/
cfg/        debug.py       kernels/  nautilus/  train.py


# Test for HD Training

In [1]:
#%tb

import importlib
import argparse
from omegaconf import OmegaConf
import os.path as osp
from datasets.inference_dataset import *
from datasets import *
import torch 

args = {'source': 'nuscenes', 'target': 'semantickitti', 'cluster_cfg': './cfg/clust_cfg/cluster_20.yaml', 'model_cfg': './cfg/model_cfg/kp_sk_infer.yaml', 'data_cfg_path': './cfg/data_cfg', 'subsample': 1, 'save_pred_path': '/root/main/3DLabelProp/results_3DLabelProp'}

cfg = OmegaConf.create(args)
cluster_cfg = OmegaConf.load(cfg.cluster_cfg)
model_cfg = OmegaConf.load(cfg.model_cfg)
cfg = OmegaConf.merge(cfg,cluster_cfg,model_cfg)
if __name__ == "__main__":
    #Get info relative to the set
    if cfg.source == "semantickitti":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set = SemanticKITTI(source_data_cfg,'train')
    elif cfg.source == "nuscenes":
        source_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set = nuScenes(source_data_cfg,'train')
    else:
        raise  NameError('source dataset not supported')

    if cfg.target == "semantickitti":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti.yaml"))
        train_set_2 = SemanticKITTI(target_data_cfg,'train')
    elif cfg.target == "nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"nuscenes.yaml"))
        train_set_2 = nuScenes(target_data_cfg,'train')
    elif cfg.target == "semanticposs":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semanticposs.yaml"))
        train_set_2 = SemanticPOSS(target_data_cfg,'train')
    elif cfg.target == "semantickitti-nuscenes":
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,"semantic-kitti-nuscenes.yaml"))
        train_set_2 = SemanticKITTI_Nuscenes(target_data_cfg,'train')
    elif "pandaset" in cfg.target:
        target_data_cfg = OmegaConf.load(osp.join(cfg.data_cfg_path,cfg.target+".yaml"))
        train_set_2 = Pandaset(target_data_cfg,'train')
    
    else:
        raise  NameError('target dataset not supported')

    #Get info relative to the model
    if cfg.architecture.model == "KPCONV":
        module = importlib.import_module('models.kpconv.kpconv')
        model_information = getattr(module, cfg.architecture.type)()
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        from models.kpconv_model import SemanticSegmentationModel
        module = importlib.import_module('models.kpconv.architecture')
        model_type = getattr(module, cfg.architecture.type)
        model = SemanticSegmentationModel(model_information,cfg,model_type)
    elif cfg.architecture.model == "SPVCNN":
        module = importlib.import_module('models.spvcnn.spvcnn')
        model_information = getattr(module, cfg.architecture.type)
        model_information.num_classes = train_set.get_n_label()
        model_information.ignore_label = -1
        model_information.in_features_dim = model_cfg.architecture.n_features
        from models.spvcnn_model import SemanticSegmentationSPVCNNModel
        model = SemanticSegmentationSPVCNNModel(model_information,cfg)
    else:
        raise  NameError('model not supported')

    output_dataset = InferenceDataset(cfg,train_set,train_set_2,model,model_information)
    #try:
    #    ius, miu = valid_dataset.compute_results()
    #except:
    output_dataset.compute_dataset()
    ius, miu = output_dataset.compute_results() # The results are already there?
    print(ius)
    print(miu)

Model ready
Sequence:  ['00', '01', '02', '03', '04', '05', '06', '07', '09', '10']


Processing dataset semantickitti:   0%|                                                          | 0/10 [00:00<?, ?it/s]

Last:  004538.bin
Last:  004538



Sequence: 00, subsample number 1/1:   0%|                                                      | 0/4538 [00:00<?, ?it/s]

Total_Pred
X_init:  5
torch.Size([1927, 3])
X_interme_enc:  torch.Size([1927, 64])
X_interme_enc:  torch.Size([1927, 128])
X_interme_enc:  torch.Size([1858, 128])
X_interme_enc:  torch.Size([1858, 256])
X_interme_enc:  torch.Size([1858, 256])
X_interme_enc:  torch.Size([1496, 256])
X_interme_enc:  torch.Size([1496, 512])
X_interme_enc:  torch.Size([1496, 512])
X_interme_enc:  torch.Size([900, 512])
X_interme_enc:  torch.Size([900, 1024])
X_interme_enc:  torch.Size([900, 1024])
X_interme_enc:  torch.Size([442, 1024])
X_interme_enc:  torch.Size([442, 2048])
X_interme_dec:  torch.Size([900, 2048])
X_interme_dec:  torch.Size([900, 1024])
X_interme_dec:  torch.Size([1496, 1024])
X_interme_dec:  torch.Size([1496, 512])
X_interme_dec:  torch.Size([1858, 512])
X_interme_dec:  torch.Size([1858, 256])
X_interme_dec:  torch.Size([1927, 256])
X_interme_dec:  torch.Size([1927, 128])
X_fin:  torch.Size([1927, 128])
X_fin:  torch.Size([1927, 16])
tensor([[ 4.4736e-01, -2.8090e-01,  3.0879e-01,  ..., 

X_interme_enc:  torch.Size([595, 512])
X_interme_enc:  torch.Size([595, 1024])
X_interme_enc:  torch.Size([595, 1024])
X_interme_enc:  torch.Size([212, 1024])
X_interme_enc:  torch.Size([212, 2048])
X_interme_dec:  torch.Size([595, 2048])
X_interme_dec:  torch.Size([595, 1024])
X_interme_dec:  torch.Size([1561, 1024])
X_interme_dec:  torch.Size([1561, 512])
X_interme_dec:  torch.Size([3458, 512])
X_interme_dec:  torch.Size([3458, 256])
X_interme_dec:  torch.Size([5562, 256])
X_interme_dec:  torch.Size([5562, 128])
X_fin:  torch.Size([5562, 128])
X_fin:  torch.Size([5562, 16])
tensor([[-5.1417e-02, -1.0184e-01, -2.1490e-02,  ...,  1.0251e+01,
          2.0507e+00,  1.1569e+00],
        [-6.2482e-02, -7.9185e-02, -2.7664e-02,  ...,  9.8578e+00,
          1.8838e+00,  1.0273e+00],
        [-5.9555e-02, -5.9253e-02, -5.9865e-03,  ...,  9.7979e+00,
          1.8632e+00,  1.2123e+00],
        ...,
        [-2.1579e-02, -1.2385e-01, -1.8861e-04,  ...,  1.2093e+01,
          3.3518e+00,  1.777

X_init:  5
torch.Size([8076, 3])
X_interme_enc:  torch.Size([8076, 64])
X_interme_enc:  torch.Size([8076, 128])
X_interme_enc:  torch.Size([4745, 128])
X_interme_enc:  torch.Size([4745, 256])
X_interme_enc:  torch.Size([4745, 256])
X_interme_enc:  torch.Size([2227, 256])
X_interme_enc:  torch.Size([2227, 512])
X_interme_enc:  torch.Size([2227, 512])
X_interme_enc:  torch.Size([900, 512])
X_interme_enc:  torch.Size([900, 1024])
X_interme_enc:  torch.Size([900, 1024])
X_interme_enc:  torch.Size([297, 1024])
X_interme_enc:  torch.Size([297, 2048])
X_interme_dec:  torch.Size([900, 2048])
X_interme_dec:  torch.Size([900, 1024])
X_interme_dec:  torch.Size([2227, 1024])
X_interme_dec:  torch.Size([2227, 512])
X_interme_dec:  torch.Size([4745, 512])
X_interme_dec:  torch.Size([4745, 256])
X_interme_dec:  torch.Size([8076, 256])
X_interme_dec:  torch.Size([8076, 128])
X_fin:  torch.Size([8076, 128])
X_fin:  torch.Size([8076, 16])
tensor([[ 3.4022e-01, -1.5639e-01, -6.2331e-02,  ...,  9.0035e+00


Processing dataset semantickitti:   0%|                                                          | 0/10 [00:06<?, ?it/s]


Exception: Just one for now